## Setup

Run this notebook from a local clone of the [svm-gmu](https://github.com/SushrutGaikwad/svm-gmu) repository. From the repo root, install the project and launch Jupyter with [uv](https://docs.astral.sh/uv/):

```bash
uv sync
uv run jupyter lab
```

The next cell puts the repo's `src/` and `experiments/` folders on the import path, so `svm_gmu` and the shared helper modules (`_common` and the `*_figs` files) import from your local clone.

# Learning the Uncertainty from Data via EM

This experiment draws samples from each per-sample true GMM, re-learns a GMM per sample using Expectation Maximization with the number of components chosen by BIC, labels each by its class, then fits SVM-GMU on the learned mixtures. As the number of samples per point grows, the learned boundary converges to the SVM-GMU boundary fit on the true GMMs. This mirrors the realistic pipeline where uncertainty is estimated from sensor data rather than given.

In [ ]:
import sys
from pathlib import Path


def _add_repo_to_path():
    """Put the repo's experiments/ and src/ folders on sys.path.

    Locates the svm-gmu repository by searching upward from the working
    directory, so the notebook runs from any local clone regardless of where
    Jupyter was started.
    """
    for base in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (base / "experiments" / "_common.py").exists():
            exp, src = base / "experiments", base / "src"
        elif (base / "_common.py").exists():
            exp, src = base, base.parent / "src"
        else:
            continue
        for path in (str(exp), str(src)):
            if path not in sys.path:
                sys.path.insert(0, path)
        return
    raise RuntimeError(
        "Could not find the svm-gmu repository. Run this notebook from a local "
        "clone (for example, `uv run jupyter lab` from the repo root)."
    )


_add_repo_to_path()

import _common as C
import em_fitted_gmu_figs as E

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.mixture import GaussianMixture

from svm_gmu import SvmGmu
from svm_gmu.plotting import plot_uncertainty

## 1. Dataset

The same 6-point banana/crescent dataset as the other experiments.

In [ ]:
X, y = C.X, C.y

In [ ]:
sample_uncertainty = C.SAMPLE_UNCERTAINTY

## 2. Reference SVM-GMU boundary (true GMMs)

In [ ]:
w_ref, b_ref = C.fit_gmu(X, y, sample_uncertainty)
print('reference w =', w_ref, ' b =', round(b_ref, 4))

## 3. Sampler, BIC-EM fit, and boundary metrics

In [ ]:
MASTER_SEED = C.MASTER_SEED
M_CANDIDATES = C.M_CANDIDATES
N_INIT = C.N_INIT

# C.sample_from_gmm, C.fit_gmm_bic, C.boundary_metrics_2d, C.fit_gmu come from _common.
# E = em_fitted_gmu_figs: E.run_sweep_multiseed runs the full multi-seed sweep.

# _draw_line is used by Sections 6 and 7; keep it local to this notebook.
def _draw_line(ax, w, b, xlim, **kwargs):
    """Draw the line w^T x + b = 0 within the given x-limits."""
    xs = np.linspace(xlim[0], xlim[1], 400)
    if abs(w[1]) > 1e-8:
        ys = -(w[0] * xs + b) / w[1]
        return ax.plot(xs, ys, **kwargs)[0]
    return ax.axvline(-b / w[0], **kwargs)

## 4. Sweep over sample size N

In [ ]:
N_VALUES = [50, 100, 250, 500, 1000, 2000, 5000, 20000]

master_rng = np.random.default_rng(MASTER_SEED)
records = []
for n in N_VALUES:
    draw_seed = int(master_rng.integers(0, 2**31 - 1))
    rng = np.random.default_rng(draw_seed)

    su_hat = []
    X_parts, y_parts = [], []
    for i, gmm in enumerate(sample_uncertainty):
        pts = C.sample_from_gmm(gmm, n, rng)
        su_hat.append(C.fit_gmm_bic(pts, M_CANDIDATES, N_INIT, seed=draw_seed))
        X_parts.append(pts)
        y_parts.append(np.full(n, y[i], dtype=np.float64))
    X_exp = np.vstack(X_parts)
    y_exp = np.concatenate(y_parts)

    w_n, b_n = C.fit_gmu(X, y, su_hat)
    angle, offset, rms = C.boundary_metrics_2d(w_n, b_n, w_ref, b_ref)
    m_chosen = [len(s["weights"]) for s in su_hat]

    records.append({
        "n": n, "n_total": 6 * n, "su_hat": su_hat,
        "X_exp": X_exp, "y_exp": y_exp,
        "w": w_n, "b": b_n,
        "angle": angle, "offset": offset, "rms": rms, "m_chosen": m_chosen,
    })
    print(f"N={n:>6d}  n_total={6 * n:>7d}  M={m_chosen}  "
          f"angle={angle:7.3f}  offset={offset:.4f}  rms={rms:.4f}")

## 5. Convergence metrics

In [ ]:
Ns = np.array([r["n"] for r in records], dtype=float)
angles = np.array([r["angle"] for r in records])
offsets = np.array([r["offset"] for r in records])
rms = np.array([r["rms"] for r in records])

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(Ns, angles, marker="o", color="#2563eb")
axes[0].set_ylabel("Angle between normals (degrees)")
axes[0].set_title("Normal-vector disagreement")
axes[1].plot(Ns, offsets, marker="o", color="#059669")
axes[1].set_ylabel(r"$\left|\frac{b_N}{\|\mathbf{w}_N\|} - \frac{b_{\mathrm{ref}}}{\|\mathbf{w}_{\mathrm{ref}}\|}\right|$")
axes[1].set_title("Offset disagreement")
axes[2].plot(Ns, rms, marker="o", color="#dc2626")
axes[2].set_ylabel("Grid RMS of decision-function difference")
axes[2].set_title("Overall boundary disagreement")
for ax in axes:
    ax.set_xscale("log")
    ax.set_xlabel(r"$N$ (samples per GMM)")
    ax.grid(True, which="both", alpha=0.3)
fig.tight_layout()
plt.show()

## 6. Boundaries at each N

In [ ]:
n_rows, n_cols = 4, 2
fig, axes = plt.subplots(n_rows, n_cols, figsize=(6.5 * n_cols, 5.0 * n_rows))
axes = np.array(axes).reshape(n_rows, n_cols)

xlim, ylim = (-5.5, 5.5), (-4.0, 5.5)
ref_color, learned_color = "#059669", "black"

for k, r in enumerate(records):
    ax = axes[k // n_cols, k % n_cols]
    plot_uncertainty(
        X, y, r["su_hat"], sigmas=(3,),
        title=rf"$N = {r['n']}$  $\left(n = {r['n_total']}\right)$",
        ax=ax,
    )
    _draw_line(ax, w_ref, b_ref, xlim, color=ref_color, linewidth=2.2,
               linestyle="--", zorder=4, label="SVM-GMU (true GMMs, reference)")
    _draw_line(ax, r["w"], r["b"], xlim, color=learned_color, linewidth=2.0,
               linestyle="-", zorder=5, label="SVM-GMU (EM-fitted GMMs)")
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)

    handles, labels = ax.get_legend_handles_labels()
    keep = {"Class +1", "Class −1",
            "SVM-GMU (true GMMs, reference)", "SVM-GMU (EM-fitted GMMs)"}
    seen, uniq = set(), []
    for h, l in zip(handles, labels):
        if l in keep and l not in seen:
            seen.add(l)
            uniq.append((h, l))
    ax.legend([h for h, _ in uniq], [l for _, l in uniq],
              loc="upper left", fontsize=8, framealpha=0.9)

for k in range(len(records), n_rows * n_cols):
    axes[k // n_cols, k % n_cols].axis("off")
fig.tight_layout()
plt.show()

## 7. Boundaries at each N over the sampled clouds

The same per-N panels as Section 6, but instead of the six observed points we overlay the sampled cloud that each EM fit was trained on (subsampled for readability). The contours are the EM-fitted (learned) GMMs, so they differ across panels and across the six clusters. As N grows, the cloud fills out the banana and crescent shapes, the learned mixtures sharpen toward them, and the EM-fitted SVM-GMU boundary (black) pulls onto the true-GMM reference (green).

In [ ]:
# Same per-N panels as above, but overlay the sampled cloud each EM fit was
# trained on instead of the six observed points. The contours are the EM-fitted
# (learned) GMMs for that N, so they differ across panels and across the six
# clusters; the observed points are not shown.
SCATTER_CAP = 800
scatter_rng = np.random.default_rng(MASTER_SEED + 1)

n_rows, n_cols = 4, 2
fig, axes = plt.subplots(n_rows, n_cols, figsize=(6.5 * n_cols, 5.0 * n_rows))
axes = np.array(axes).reshape(n_rows, n_cols)

xlim, ylim = (-5.5, 5.5), (-4.0, 5.5)
ref_color, learned_color = "#059669", "black"

for k, r in enumerate(records):
    ax = axes[k // n_cols, k % n_cols]

    # Layer 1: EM-fitted GMM contours for this N. plot_uncertainty also
    # scatters the six observed points; strip those PathCollections so only
    # the learned contours remain.
    plot_uncertainty(
        X, y, r["su_hat"], sigmas=(3,),
        title=rf"$N = {r['n']}$  $\left(n = {r['n_total']}\right)$",
        ax=ax,
    )
    for coll in list(ax.collections):
        if isinstance(coll, plt.matplotlib.collections.PathCollection):
            coll.remove()

    # Layer 2: the sampled cloud the EM fit was trained on (subsampled).
    X_exp, y_exp = r["X_exp"], r["y_exp"]
    if X_exp.shape[0] > SCATTER_CAP:
        idx = scatter_rng.choice(X_exp.shape[0], size=SCATTER_CAP, replace=False)
        X_plot, y_plot = X_exp[idx], y_exp[idx]
    else:
        X_plot, y_plot = X_exp, y_exp
    for label, color, marker, class_label in [
        (+1, "#2563eb", "o", "Class +1"),
        (-1, "#dc2626", "s", "Class −1"),
    ]:
        mask = y_plot == label
        ax.scatter(
            X_plot[mask, 0], X_plot[mask, 1],
            s=30, c=color, marker=marker,
            alpha=0.5, edgecolors="none", zorder=2,
            label=class_label,
        )
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)

    # Layer 3: reference and EM-fitted SVM-GMU boundaries.
    _draw_line(ax, w_ref, b_ref, xlim, color=ref_color, linewidth=2.2,
               linestyle="--", zorder=4, label="SVM-GMU (true GMMs, reference)")
    _draw_line(ax, r["w"], r["b"], xlim, color=learned_color, linewidth=2.0,
               linestyle="-", zorder=5, label="SVM-GMU (EM-fitted GMMs)")
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)

    handles, labels = ax.get_legend_handles_labels()
    keep = {"Class +1", "Class −1",
            "SVM-GMU (true GMMs, reference)", "SVM-GMU (EM-fitted GMMs)"}
    seen, uniq = set(), []
    for h, l in zip(handles, labels):
        if l in keep and l not in seen:
            seen.add(l)
            uniq.append((h, l))
    ax.legend([h for h, _ in uniq], [l for _, l in uniq],
              loc="upper left", fontsize=8, framealpha=0.9)

for k in range(len(records), n_rows * n_cols):
    axes[k // n_cols, k % n_cols].axis("off")
fig.tight_layout()
plt.show()

## 8. Interpretation

As N grows, the BIC-selected component counts climb from mostly one toward the true five or six, the EM-fitted contours sharpen into the true banana and crescent shapes, and the learned boundary (black) pulls onto the reference (green). The offset and angle metrics fall steadily toward zero. SVM-GMU therefore degrades gracefully when the mixtures are estimated rather than given: collect repeated noisy measurements, fit a GMM per example, and feed those to SVM-GMU.